In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import optuna
import shap
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import category_encoders as ce

# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading and preparing data")
print("=" * 60)

FILE_PATH = r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\NEW FIXES\microrna_with_families_all.xlsx'
df = pd.read_excel(FILE_PATH)

df['parasite'] = df['parasite'].str.replace(r'\s+', '', regex=True)

# ── Build lookup dicts BEFORE dropping columns ────────────────
# IMPORTANT: include family_conservation in the lookup so the
# Streamlit app can display it correctly at prediction time.
df_lookup = df.copy()

mirna_lookup = (
    df_lookup[[
        'microrna', 'microrna_group_simplified',
        'seed_family', 'mirbase_accession', 'family_conservation'
    ]]
    .drop_duplicates('microrna')
    .set_index('microrna')
    .to_dict('index')
)

accession_lookup = (
    df_lookup[[
        'mirbase_accession', 'microrna',
        'microrna_group_simplified', 'seed_family', 'family_conservation'
    ]]
    .dropna(subset=['mirbase_accession'])
    .drop_duplicates('mirbase_accession')
    .set_index('mirbase_accession')
    .to_dict('index')
)

print(f"miRNA lookup     : {len(mirna_lookup)} unique miRNA names")
print(f"Accession lookup : {len(accession_lookup)} unique accession numbers")

# Verify family_conservation is in lookup
sample = list(mirna_lookup.values())[0]
print(f"Sample lookup entry: {sample}")

# Drop columns not used as features
cols_to_drop = ['microrna', 'infection', 'microrna_group_simplified', 'mirbase_accession']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# NaN for unknown seed families
df['seed_family'] = df['seed_family'].replace(['not_broadly_conserved', 'not_found'], np.nan)

# is_conserved flag
df['is_conserved'] = df['seed_family'].notna().astype(int)

# family_conservation: already numeric (2.0, 1.0, 0.0, -1.0, NaN)
# NaN rows → LightGBM will rely on other features for those rows
df['family_conservation'] = pd.to_numeric(df['family_conservation'], errors='coerce')

# Interaction feature
df['parasite_celltype'] = df['parasite'].astype(str) + '_' + df['cell type'].astype(str)

print(f"Total rows            : {len(df)}")
print(f"Seed family known     : {df['is_conserved'].sum()}")
print(f"Seed family unknown   : {(df['is_conserved']==0).sum()}")
print(f"family_conservation values: {sorted(df['family_conservation'].dropna().unique())}")
print(f"Target balance        :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Features and target
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Features and target")
print("=" * 60)

TARGET   = 'is_upregulated'
CAT_COLS = ['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']
NUM_COLS = ['time', 'is_conserved', 'family_conservation']

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Categorical   : {CAT_COLS}")
print(f"Numeric       : {NUM_COLS}")
print(f"Total features: {len(CAT_COLS) + len(NUM_COLS)}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Optuna hyperparameter search
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Optuna hyperparameter search (30 trials)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 20),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight':      'balanced',
        'random_state':      42,
        'verbose':           -1,
    }
    pipe_opt = Pipeline([
        ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
        ('model',   LGBMClassifier(**params))
    ])
    return cross_val_score(pipe_opt, X, y, cv=cv, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
print("Running 30 trials... Please wait.")
study.optimize(objective, n_trials=30)

BEST_PARAMS = study.best_params
BEST_PARAMS.update({'class_weight': 'balanced', 'random_state': 42, 'verbose': -1})

print(f"\nBest AUC (Optuna): {study.best_value:.4f}")
print("Best params:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")


# ─────────────────────────────────────────────────────────────
# STEP 4 — Cross-Validation with best params
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Cross-Validation (5-fold)")
print("=" * 60)

pipe = Pipeline([
    ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
    ('model',   LGBMClassifier(**BEST_PARAMS))
])

auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
f1  = cross_val_score(pipe, X, y, cv=cv, scoring='f1')

print(f"ROC-AUC : {auc.mean():.3f} ± {auc.std():.3f}")
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"F1      : {f1.mean():.3f} ± {f1.std():.3f}")
print(f"AUC per fold: {[round(x, 3) for x in auc.tolist()]}")


# ─────────────────────────────────────────────────────────────
# STEP 5 — Train final model on all data
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Training final model on all data")
print("=" * 60)

pipe.fit(X, y)
print("Done.")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Permutation Feature Importance
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Permutation Feature Importance")
print("=" * 60)

X_enc = pipe.named_steps['encoder'].transform(X)
perm  = permutation_importance(
    pipe.named_steps['model'],
    X_enc, y, n_repeats=30, random_state=42, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature':    X.columns.tolist(),
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>10} {'Std':>8}")
print("-" * 47)
for _, row in perm_df.iterrows():
    bar  = '█' * max(0, int(row['importance'] * 60))
    sign = '+' if row['importance'] >= 0 else ''
    print(f"  {row['feature']:<23} {sign}{row['importance']:.4f} ± {row['std']:.4f}  {bar}")


# ─────────────────────────────────────────────────────────────
# STEP 7 — Build SHAP explainer
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Building SHAP explainer")
print("=" * 60)

explainer = shap.TreeExplainer(pipe.named_steps['model'])
shap_vals = explainer.shap_values(X_enc)
print(f"SHAP shape: {np.array(shap_vals).shape}  ✓")


# ─────────────────────────────────────────────────────────────
# STEP 8 — Save model bundle
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8 — Saving model bundle")
print("=" * 60)

model_bundle = {
    'model':     pipe,
    'encoder':   pipe.named_steps['encoder'],
    'lgbm':      pipe.named_steps['model'],
    'explainer': explainer,
    'metrics': {
        'auc_mean':   round(auc.mean(), 3),
        'auc_std':    round(auc.std(),  3),
        'acc_mean':   round(acc.mean(), 3),
        'acc_std':    round(acc.std(),  3),
        'f1_mean':    round(f1.mean(),  3),
        'f1_std':     round(f1.std(),   3),
        'auc_folds':  [round(x, 3) for x in auc.tolist()],
        'best_params': BEST_PARAMS,
        'n_train':    len(X),
        'feature_importance': perm_df.to_dict('records'),
    },
    'options': {
        'parasite':  sorted(df['parasite'].dropna().unique().tolist()),
        'organism':  sorted(df['organism'].dropna().unique().tolist()),
        'cell_type': sorted(df['cell type'].dropna().unique().tolist()),
        'time':      sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df['seed_family'].dropna().unique().tolist()),
    },
    'cat_cols':      CAT_COLS,
    'feature_names': list(X.columns),
    'mirna_lookup':      mirna_lookup,
    'accession_lookup':  accession_lookup,
}

BUNDLE_PATH = r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\NEW FIXES\Model206_ALL_model.pkl'
with open(BUNDLE_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)
print(f"Saved: {BUNDLE_PATH}")

STEP 1 — Loading and preparing data
miRNA lookup     : 115 unique miRNA names
Accession lookup : 98 unique accession numbers
Sample lookup entry: {'microrna_group_simplified': 'let-7a', 'seed_family': 'let-7-5p/98-5p', 'mirbase_accession': 'MIMAT0000062', 'family_conservation': 2.0}
Total rows            : 206
Seed family known     : 198
Seed family unknown   : 8
family_conservation values: [np.float64(-1.0), np.float64(0.0), np.float64(1.0), np.float64(2.0)]
Target balance        :
is_upregulated
1    103
0    103

STEP 2 — Features and target
Categorical   : ['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']
Numeric       : ['time', 'is_conserved', 'family_conservation']
Total features: 8

STEP 3 — Optuna hyperparameter search (30 trials)
Running 30 trials... Please wait.

Best AUC (Optuna): 0.9355
Best params:
  n_estimators: 173
  max_depth: 7
  learning_rate: 0.06226295614570949
  num_leaves: 26
  min_child_samples: 6
  subsample: 0.6980632071909614
  colsam